# Medical dictation -> SOAP note

This notebook goes through the whole assignment in order:

1. **Part A - transcription.** `sample_dictation.mp3` is transcribed locally with **faster-whisper (Whisper large-v3)**.
2. **Part B - SOAP note.** The transcript is turned into a structured SOAP note by **Llama 3.1 8B running locally in Ollama**, using a JSON schema so the output is always valid JSON.
3. **Bonus - checks.**
   - A two-signal check that moves anything patient-reported out of *Objective*.
   - A check that every number in the note really appears in the transcript.

Everything runs on my laptop: no audio or text is sent to a cloud API. That's what I'd want for real patient data anyway.
The actual logic lives in the `soap_pipeline/` package. This notebook calls it step by step and shows what happens in between.

**Hardware I ran this on:** a laptop with 8 GB RAM, a GTX 1650 (4 GB) and a 16-thread CPU.
- Whisper ran on the CPU (int8). It takes about 13 min for the 13.75-min recording.
- Llama 3.1 8B runs partly on the GPU through Ollama.

In [1]:
import json, platform, time
from pathlib import Path

import requests
import faster_whisper
from IPython.display import Markdown, display

from soap_pipeline.transcribe import low_confidence_segments, repeated_phrases
from soap_pipeline.soap import (MODEL, SOAP_SCHEMA, SYSTEM_PROMPT, build_user_prompt,
                                generate_soap)
from soap_pipeline.checks import (rule_flags, llm_flags, fix_subjective_in_objective,
                                  ungrounded_numbers)
from soap_pipeline.render import to_markdown, save_note

AUDIO = "data/sample_dictation.mp3"
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

print("Python        ", platform.python_version())
print("faster-whisper", faster_whisper.__version__)
print("Ollama        ", requests.get("http://localhost:11434/api/version", timeout=5).json()["version"])
print("LLM           ", MODEL)

Python         3.12.10
faster-whisper 1.2.1


Ollama         0.40.16
LLM            llama3.1:8b


## Part A - Transcription

**Why faster-whisper + large-v3:**
- Whisper large-v3 is the most accurate open Whisper checkpoint, and it handles medical vocabulary well ("lateral epicondylitis", "pronate and supinate", "NSAIDs").
- faster-whisper is the same model re-implemented on CTranslate2. It's roughly 4x faster and supports int8, which is what lets large-v3 run on a CPU at all.

**Decoding settings** (all are explained in `soap_pipeline/transcribe.py`):

| setting | value | why |
|---|---|---|
| `language` | `"en"` | no auto-detect guess on the first 30 s |
| `beam_size` | 5 | most likely wording, not greedy |
| `temperature` | 0 -> 1.0 fallback | starts deterministic; a window that looks repetitive or low-confidence is decoded again |
| `condition_on_previous_text` | `False` | stops one bad window from looping into the next (see below) |
| `vad_filter` | `True` | skips silences, where Whisper tends to invent text |
| `initial_prompt` | none | a vocabulary prompt can make the model "hear" words that were never said |

**What went wrong first:** my first run used `temperature=0` and `condition_on_previous_text=True`, and large-v3 got stuck in loops.
- Example: *"so it's also a case of something called lateral epicondylitis"* was repeated **8 times**.
- `temperature=0` alone switches off Whisper's own repetition safety net, and conditioning lets a loop carry into the next window.
- Changing those two settings fixed it.
- I also added `repeated_phrases()`, which flags any 3-15 word phrase repeated 3+ times in a row, so a loop can't slip through silently again.

In [2]:
RUN_TRANSCRIPTION = True   # set to False to reuse outputs/transcript_segments.json

if RUN_TRANSCRIPTION:
    # Part A runs in its own process: when it exits, the OS gets Whisper's ~2 GB back.
    # In-process, even after `del model`, the kernel kept the memory and on my 8 GB
    # laptop the LLM call that follows ran out of RAM and crawled.
    import subprocess, sys
    run = subprocess.run([sys.executable, "-m", "soap_pipeline.transcribe", AUDIO],
                         capture_output=True, text=True)
    print(run.stdout.strip())
    run.check_returncode()

result = json.loads((OUT / "transcript_segments.json").read_text(encoding="utf-8"))
transcript = result["text"]
print(f"Audio length : {result['duration_sec'] / 60:.2f} min")
print(f"Segments     : {len(result['segments'])}")
print(f"Words        : {len(transcript.split())}")

GPU not usable (RuntimeError: Library cublas64_12.dll is not found or cannot be loaded), using CPU
Transcribed 825.02 s of audio in 1244 s, 101 segments
Audio length : 13.75 min
Segments     : 101
Words        : 1556


### Quality checks on the transcript

- **Repetition loops:** should be empty.
- **Low-confidence segments:** `avg_logprob < -0.6`, meaning parts a human should listen to again.

In [3]:
print("Repetition loops     :", repeated_phrases(transcript) or "none")
unsure = low_confidence_segments(result)
print("Low-confidence segs  :", len(unsure))
for s in unsure:
    print(f"  [{s['start']:7.2f}s] {s['avg_logprob']}  {s['text']}")

Repetition loops     : none
Low-confidence segs  : 0


Here is the start of the transcript with timestamps (the full text is saved to `outputs/transcript.txt`):

In [4]:
for s in result["segments"][:12]:
    print(f"[{s['start']:7.2f} - {s['end']:7.2f}]  {s['text']}")

[   0.00 -   10.06]  what brings you in hi i've i've had this pain on the outside of my uh right elbow now it's it's
[  10.06 -   16.54]  i first started to knowing noticing it several months ago but uh recently it's just been more
[  16.54 -   25.14]  painful okay so you said several months ago um did anything happened several months ago was there
[  25.14 -   33.60]  any sort of trigger trauma anything like that to that area no there wasn't any any trauma or
[  33.60 -   43.64]  any uh triggers that that I noticed I was just um feeling it uh a bit more at the end of of work
[  43.64 -   49.98]  um uh yeah I was just having a feeling the pain a bit more at the end of work
[  50.76 -   54.64]  Okay. Does anything make it better or worse, the pain?
[  57.58 -   64.42]  Yeah, really if I'm just resting the elbow, it makes it better.
[  64.42 -   70.90]  And I've tried things like ibuprofen, which has helped with the pain.
[  71.10 -   79.08]  I'll do that for helping get through work som

In [5]:
print(transcript)

what brings you in hi i've i've had this pain on the outside of my uh right elbow now it's it's i first started to knowing noticing it several months ago but uh recently it's just been more painful okay so you said several months ago um did anything happened several months ago was there any sort of trigger trauma anything like that to that area no there wasn't any any trauma or any uh triggers that that I noticed I was just um feeling it uh a bit more at the end of of work um uh yeah I was just having a feeling the pain a bit more at the end of work Okay. Does anything make it better or worse, the pain? Yeah, really if I'm just resting the elbow, it makes it better. And I've tried things like ibuprofen, which has helped with the pain. I'll do that for helping get through work sometimes if the pain is bad enough. Right. Okay. Um, and if you were to describe the quality of the pain, is it sharp, throbby, achy? Uh, it's, um, kind of, uh, well, it's achy. And then sometimes depending on th

## Part B - SOAP note with a local LLM

**Why Llama 3.1 8B in Ollama:**
- It's the strongest instruction-following model that still runs on a 4 GB GPU plus system RAM.
- Ollama gives it **structured outputs**: I pass a JSON schema and the model's tokens are constrained to it, so the result is always valid JSON in the right shape. There's no regex parsing and no "please answer in JSON" hoping.

**How I got to the final prompt** (each step came from reading the output against the transcript):

1. **Clear S / O / A / P definitions.** The recording is a conversation without speaker labels, so the prompt explains how to tell clinician from patient.
   - Results of an exam the *clinician directs* (e.g. "press on the outside of the elbow") are **Objective findings**.
   - The patient's own history stays **Subjective**.
2. **Accuracy rules:**
   - Use only facts that were said.
   - Keep drug names and the clinician's wording for movements exactly as spoken.
   - Record negatives.
   - Write "Not documented" instead of guessing.
   - Don't add diagnoses the clinician didn't state.
3. **A detailed schema used as a checklist.**
   - My first schema had one big "HPI" list and one "physical exam" list. The 8B model summarised and dropped roughly a third of the facts. It also turned "should resolve in 6 months to 2 years" into a *follow-up appointment*.
   - With specific keys (`aggravating_factors`, `relieving_factors`, `inspection`, `special_tests_and_provocative_maneuvers`, `investigations`, `patient_education_and_expected_course`...), the model has to stop and look for each one.
4. **Single pass, no chunking - tried and rejected.**
   - I tested splitting the transcript into ~600-word chunks and merging the per-chunk notes. It recovered a couple of details, but it was clearly worse overall.
   - A chunk that only covers, say, the review of systems still has to fill every field, and the 8B model started filling them with plausible guesses. It copied a pain score from the prompt's own example and invented "follow up in 2 weeks".
   - In a clinical note an invented fact is far worse than a missing one, so I kept one pass over the whole transcript (about 2k tokens, well inside the 8k context).
5. **Examples in the prompt are not from this recording** (knee, hip...), so the prompt isn't tuned to the test case. After the chunking experiment I also removed all numbers from the examples.
6. **`temperature=0` and a fixed `seed`**, so the same transcript gives the same note.

In [6]:
print(SYSTEM_PROMPT)

You are a clinical documentation specialist. You convert the transcript of a clinical encounter (a clinician talking with a patient) into a SOAP note. The transcript has no speaker labels, so work out who is speaking from context: the clinician asks questions and gives instructions, the patient answers. You never add, guess or infer anything that was not said.

Section rules:
- SUBJECTIVE = what the PATIENT tells the clinician: the complaint and its history, what they have tried, past medical and surgical history, medications, allergies, immunizations, family history, social history (work, smoking, alcohol, drugs, travel, sick contacts) and the review of systems (symptoms asked about, present or denied).
  Past medical history = ongoing / chronic conditions, or that the patient says they are otherwise healthy. Surgeries and hospitalizations go in past surgical history.
- OBJECTIVE = findings from the clinician's examination: vital signs, inspection, palpation, range of motion, special 

In [7]:
t0 = time.time()
note = generate_soap(transcript)
print(f"LLM took {time.time() - t0:.0f} s")
print(json.dumps(note, indent=2))

LLM took 1307 s
{
  "patient": {
    "name": "Not documented",
    "age": "Not documented",
    "sex": "Not documented",
    "date_of_service": "Not documented"
  },
  "subjective": {
    "chief_complaint": "Pain on the outside of the right elbow",
    "history_of_present_illness": {
      "onset_and_duration": [
        "several months ago",
        "getting worse"
      ],
      "location_and_radiation": [
        "outside of the right elbow",
        "no radiation"
      ],
      "quality": [
        "achy",
        "sharp"
      ],
      "severity": [
        "about a four",
        "out of 10"
      ],
      "course_over_time": [
        "getting worse"
      ],
      "aggravating_factors": [
        "repetitive movements at work",
        "bending the wrist down"
      ],
      "relieving_factors": [
        "resting the elbow"
      ],
      "treatments_tried": [
        "ibuprofen",
        "topical ointments"
      ],
      "associated_symptoms_and_pertinent_negatives": [
    

## Bonus - Logic checks

### 1. Is any Subjective information sitting in Objective?

This is harder than it looks in a conversation. For example, the patient says *"I have full range of motion of both the elbow and hand"* while answering a history question. As a sentence on its own, "Full range of motion of elbow and hand" reads exactly like an exam finding. What makes it Subjective is **who said it and when**, so the check looks at the transcript, not just the note.

Three signals for every Objective statement:

| signal | what it does | deterministic? |
|---|---|---|
| **Exam-step check** | finds the transcript passage the statement came from, then looks just before it for a clinician exam instruction ("look at your...", "palpate", "push on", "extend your...", "pronate", "I'm going to get you to...") | yes |
| **Wording rule** | patient-report wording: *reports, states, complains of, denies, feels, rates x/10, per patient* | yes |
| **LLM classifier** | a narrow second call that sees each statement **plus its transcript excerpt** and decides: answer to a history question, or result of an exam step? Two contrasting few-shot examples (not from this case) | no |

**Decision:**
- For exam subsections (inspection, palpation, range of motion, special tests), the **exam-step check decides**. An exam finding without an exam step before it came from the history, so it moves to Subjective.
- I made the deterministic check the decider because, while testing, the 8B classifier gave different answers for the same statement depending on what else was in the batch.
- The wording rule and the LLM are still run and logged. If either disagrees with the exam-step check, that's written to the audit log, so a reviewer sees it.
- For vital signs and labs (measurements, not instructions), the LLM and the wording rule decide.

In [8]:
from soap_pipeline.checks import exam_instruction_before

print(f"{'Objective statement':62} exam instruction found before it")
for sub, items in note["objective"].items():
    for text in items:
        found, cue = exam_instruction_before(text, transcript)
        print(f"{text[:60]:62} {cue if found else '(no matching passage)'}")

Objective statement                                            exam instruction found before it
no redness or swelling                                         look at that
no skin changes or deformities                                 look at that
pain at the lateral elbow                                      get you to
no pain at the medial aspect                                   get you to
no pain in the forearm or biceps                               push on
full range of motion of both the elbow and hand                None
pain with wrist extension                                      palpate
pain with wrist flexion with the elbow in full extension       get you to


To show the check really works, here it is on a copy of the note with three deliberately misplaced statements:
- with a keyword: "Patient reports ..."
- without a keyword, but grounded in the history part of the transcript: "Full range of motion of the elbow and hand"
- something that is never said in the transcript at all: "Pain wakes the patient up at night"
  - With nothing in the transcript to trace it to, the exam-step check can't decide.
  - If the LLM classifier labels it Subjective it gets moved. If not, it's flagged **needs_review** as a possible hallucination.
  - Either way it can't stay in Objective silently.

In [9]:
import copy

planted = ["Patient reports the pain is worse at the end of the work day",
           "Full range of motion of the elbow and hand",
           "Pain wakes the patient up at night"]
broken = copy.deepcopy(note)
broken["objective"]["palpation"].append(planted[0])
broken["objective"]["range_of_motion_and_strength"].append(planted[1])
broken["objective"]["inspection"].append(planted[2])

for entry in fix_subjective_in_objective(broken, transcript):
    if any(entry.get(k) in planted for k in ("moved", "needs_review", "kept")):
        print(json.dumps(entry, indent=2))

{
  "needs_review": "Pain wakes the patient up at night",
  "in": "objective.inspection",
  "reason": "could not be traced to any passage in the transcript (possible hallucination) - left in place for review"
}
{
  "moved": "Patient reports the pain is worse at the end of the work day",
  "from": "objective.palpation",
  "to": "subjective.history_of_present_illness.other_details",
  "reason": "no examination step before it in the transcript, so it came from the history + patient-report wording"
}
{
  "moved": "Full range of motion of the elbow and hand",
  "from": "objective.range_of_motion_and_strength",
  "to": "subjective.history_of_present_illness.other_details",
  "reason": "no examination step before it in the transcript, so it came from the history"
}


Now the same check on the real note. The audit log goes into the validation report:

In [10]:
moves = fix_subjective_in_objective(note, transcript)
print(f"{sum('moved' in m for m in moves)} statement(s) moved out of Objective")
for m in moves:
    print(json.dumps(m, indent=2))

1 statement(s) moved out of Objective
{
  "kept": "no skin changes or deformities",
  "in": "objective.inspection",
  "reason": "flagged (LLM classifier: clinician asks 'So no skin changes or deformities there?') but the transcript shows an exam step right before it (\"look at that\")"
}
{
  "moved": "full range of motion of both the elbow and hand",
  "from": "objective.range_of_motion_and_strength",
  "to": "subjective.history_of_present_illness.other_details",
  "reason": "no examination step before it in the transcript, so it came from the history + LLM classifier: clinician asks 'are you still able to move that arm your right arm um as well as your hand i am yeah i have full range of motion of both the the elbow and uh hand'"
}


### 2. Are all numbers in the note grounded in the transcript?

A wrong digit in a dose, a vital sign or a duration is the most dangerous thing an LLM can get wrong in a clinical note. So every number in the final note has to appear in the transcript.

Speech is transcribed as words ("about a four"), so number words in the transcript count too. The "/10" of a pain score is ignored, because the clinician defines that scale in words.

**Limitation:** this checks *that* a number was said, not *where*. During testing a run wrote a pain score of 6/10, and it passed because "six" appears elsewhere ("six months"). Catching that needs the number checked together with its context (number + nearby words). That's the next thing I'd add.

In [11]:
missing = ungrounded_numbers(note, transcript)
print("Numbers in the note that are not in the transcript:", missing or "none")

# sanity check that the guard really catches an invented number
print("Planted 500 mg ->", ungrounded_numbers({"plan": ["Ibuprofen 500 mg three times a day"]}, transcript))

Numbers in the note that are not in the transcript: none
Planted 500 mg -> [{'number': '500', 'in': 'Ibuprofen 500 mg three times a day'}]


## Final SOAP note

In [12]:
report = {
    "llm": MODEL,
    "transcript_repetition_loops": repeated_phrases(transcript),
    "subjective_in_objective_check": moves,
    "numbers_not_found_in_transcript": missing,
}
save_note(note, report)
display(Markdown(to_markdown(note)))

# SOAP Note

**Patient:** Not documented  
**Age:** Not documented  
**Sex:** Not documented  
**Date of service:** Not documented

## S - Subjective

**Chief complaint:** Pain on the outside of the right elbow

### History of present illness

**Onset and duration**
- several months ago
- getting worse

**Location and radiation**
- outside of the right elbow
- no radiation

**Quality**
- achy
- sharp

**Severity**
- about a four
- out of 10

**Course over time**
- getting worse

**Aggravating factors**
- repetitive movements at work
- bending the wrist down

**Relieving factors**
- resting the elbow

**Treatments tried**
- ibuprofen
- topical ointments

**Associated symptoms and pertinent negatives**
- no numbness or tingling
- no changes to senses
- no coughing or shortness of breath

**Functional impact**
- starting to become an issue at work

**Other details**
- right-handed
- no trauma or triggers
- full range of motion of both the elbow and hand

**Past medical history**
- C-section 17 years ago

**Past surgical history**
- C-section

**Medications**
- Tylenol
- Advil

**Allergies**
- none

**Immunizations**
- up to date

**Family history**
- no skeletal or muscular issues
- no neurological issues
- no autoimmune conditions

**Social history**
- no smoking
- occasional alcohol use
- no marijuana use

**Review of systems**
- no weight loss
- no fevers or chills
- no fatigue
- no rashes
- no changes to senses
- no coughing or shortness of breath
- no chest pain
- no nausea or vomiting
- no changes to urine
- no memory loss or confusion

## O - Objective

**Vital signs**
- Not documented

**Inspection**
- no redness or swelling
- no skin changes or deformities

**Palpation**
- pain at the lateral elbow
- no pain at the medial aspect
- no pain in the forearm or biceps

**Range of motion and strength**
- Not documented

**Special tests and provocative maneuvers**
- pain with wrist extension
- pain with wrist flexion with the elbow in full extension

**Labs and imaging results**
- Not documented

## A - Assessment

**Diagnoses**
- lateral epicondylitis (also known as tennis elbow)

**Clinical reasoning**
- repetitive strain at work and continuous contraction of the wrist extensor muscle group

## P - Plan

**Investigations**
- no imaging necessary

**Medications**
- continue taking Tylenol or NSAIDs for pain

**Therapies and referrals**
- physical therapy to increase mobility and strength in the arm

**Activity modification and self care**
- activity modifications at work
- use a compression sleeve
- apply ice to the area for pain and swelling

**Devices and supports**
- compression sleeve
- splints

**Patient education and expected course**
- should resolve in about six months to two years
- may go away in a couple months with physical therapy and compression and splints

**Follow up**
- Not documented


## Files produced

| file | what it is |
|---|---|
| `outputs/transcript.txt` | raw transcript (Part A deliverable) |
| `outputs/transcript_segments.json` | same transcript with timestamps and per-segment confidence |
| `outputs/soap_note.json` | structured SOAP note (Part B deliverable) |
| `outputs/soap_note.md` | the same note, human-readable |
| `outputs/validation_report.json` | results of the logic checks, including the audit log of anything moved |

## Limitations and what I'd do next in production

- **Speaker labels.** Whisper doesn't say who is speaking, so the LLM infers it from context. Diarization (e.g. pyannote) before the LLM step would make the S/O split more reliable.
- **Human in the loop.** Low-confidence segments and every automatic move are surfaced, not hidden. A clinician should sign off on the note.
- **Bigger model, same code.** With a GPU server, `--llm llama3.1:70b` (or any Ollama model) is a one-flag change. A larger model would mainly improve recall on long encounters.
- **Evaluation.** With a ground-truth transcript I'd track WER (e.g. with `jiwer`) and a medical-term recall list. For the note, a field-level checklist reviewed by a clinician.